# 🌫️ Anand Vihar AQI Prediction — LSTM & GRU (EDA-Enhanced)

**Dataset:** `AnandVihar_AQI_cleaned.csv`  
**Task:** Time-series regression — predict **next-hour AQI** (target = `AQI_next = AQI.shift(-1)`, no leakage)  
**Inference:** user inputs last 48 h of readings → model outputs AQI for the following hour  
**Models:** LSTM · GRU · Bidirectional variants  
**Enhancements over v1:**
- EDA-driven feature engineering (lag features, rolling statistics, cyclical time encoding)
- GRU model trained in parallel for fair comparison
- Bidirectional wrappers for both architectures
- Side-by-side model comparison dashboard
- Per-pollutant contribution analysis

---
### Notebook Structure
1. Install & Import Libraries  
2. Load & Prepare Data  
3. EDA-Driven Feature Engineering  
4. Feature Selection & Correlation Check  
5. Robust Scaling (StandardScaler)  
6. Create Sliding Window Sequences  
7. Train / Validation / Test Split  
8. Build LSTM Model  
9. Build GRU Model  
10. Train Both Models  
11. Evaluate & Compare on Test Set  
12. Visualise Predictions  
13. Classification Metrics (AQI Category)  
14. Model Comparison Dashboard  
15. Save Models & Scalers

## 1. Install & Import Libraries

In [ ]:
!pip install tensorflow scikit-learn pandas numpy matplotlib seaborn scipy joblib --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings, joblib
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from scipy import stats as sp_stats

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, GRU, Dense, Dropout, BatchNormalization, Bidirectional, Input
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# ── Plot theme ────────────────────────────────────────────────────────────────
DARK  = '#0d0f1a'
CARD  = '#141728'
ACC1  = '#00e5ff'
ACC2  = '#ff4081'
ACC3  = '#69ff47'
ACC4  = '#ffb347'
MUTED = '#8892b0'
WHITE = '#e6f1ff'

plt.rcParams.update({
    'figure.facecolor': DARK,
    'axes.facecolor':   CARD,
    'text.color':       WHITE,
    'axes.labelcolor':  MUTED,
    'xtick.color':      MUTED,
    'ytick.color':      MUTED,
    'axes.edgecolor':   '#2a2f4a',
    'grid.color':       '#1e2340',
    'grid.linestyle':   '--',
    'grid.linewidth':   0.5,
})

CAT_ORDER = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
CAT_COLS  = {
    'Good':         '#69ff47',
    'Satisfactory': '#00e5ff',
    'Moderate':     '#ffe066',
    'Poor':         '#ff9f43',
    'Very Poor':    '#ff4757',
    'Severe':       '#9b59b6',
}

def style_ax(ax, title='', xlabel='', ylabel=''):
    ax.set_facecolor(CARD)
    ax.set_title(title, color=WHITE, fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel(xlabel, color=MUTED, fontsize=9)
    ax.set_ylabel(ylabel, color=MUTED, fontsize=9)
    ax.tick_params(colors=MUTED, labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#2a2f4a')
    ax.grid(True)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow version : {tf.__version__}')
print('✅ All libraries imported')

## 2. Load & Prepare Data

In [ ]:
df = pd.read_csv(r'D:\AQI_Project_new\data\clean\AnandVihar_AQI_cleaned.csv', parse_dates=['Timestamp'])
df = df.set_index('Timestamp').sort_index()

print(f'Shape      : {df.shape}')
print(f'Date range : {df.index.min().date()}  →  {df.index.max().date()}')
print(f'Missing    : {df.isnull().sum().sum()}')
df.head(3)

## 3. EDA-Driven Feature Engineering

Based on EDA findings and feature importance analysis, we engineer additional features and **drop weak contributors** (raw Benzene, raw Toluene, BP, AQI_lag3, PM10_lag1):

| Feature Group | Features Created | Rationale |
|---|---|---|
| **Lag features** | AQI_lag1, AQI_lag2, PM25_lag1 | Strongest autocorrelation lags only (lag3 dropped — dominated by lag1/2) |
| **Rolling statistics** | AQI_roll3_mean, AQI_roll6_mean, AQI_roll12_mean | Trend capture; roll3_std dropped (low marginal value) |
| **Cyclical time** | hour_sin, hour_cos | Diurnal traffic pattern at Anand Vihar bus terminal; month encoding dropped |
| **Derived pollutants** | NOx_total, Toluene_Benzene_ratio | NOx = NO+NO2 composite; T/B ratio replaces raw Benzene & Toluene as a source fingerprint |

**Dropped from original set:** `Benzene` (raw), `Toluene` (raw), `BP`, `AQI_lag3`, `PM10_lag1`, `month_sin`, `month_cos`, `AQI_roll3_std`

In [ ]:
# ── Work on a copy ──────────────────────────────────────────────────────────
fe = df.copy()

# ── Target: AQI one hour ahead ────────────────────────────────────────────────
# Shift AQI by -1 so each row's target is the NEXT hour's AQI.
# The feature sequence [t-47 … t] represents what the user already has;
# AQI_next is what we want to predict.
fe['AQI_next'] = fe['AQI'].shift(-1)

# ── 1. Lag features (Tier 1 only: lag1, lag2; lag3 dropped) ──────────────────
fe['AQI_lag1']  = fe['AQI'].shift(1)
fe['AQI_lag2']  = fe['AQI'].shift(2)
fe['PM25_lag1'] = fe['PM25'].shift(1)
# Dropped: AQI_lag3 (dominated by lag1/lag2), PM10_lag1 (low marginal signal)

# ── 2. Rolling statistics (roll3_std dropped — low marginal value) ────────────
for w in [3, 6, 12]:
    fe[f'AQI_roll{w}_mean'] = fe['AQI'].shift(1).rolling(w).mean()

# ── 3. Cyclical hour encoding only (month dropped — hour dominates here) ──────
fe['hour_sin'] = np.sin(2 * np.pi * fe.index.hour / 24)
fe['hour_cos'] = np.cos(2 * np.pi * fe.index.hour / 24)

# ── 4. Derived pollutant features ─────────────────────────────────────────────
fe['NOx_total']             = fe['NO'] + fe['NO2']
fe['Toluene_Benzene_ratio'] = fe['Toluene'] / (fe['Benzene'] + 1e-6)
# Raw Benzene and Toluene are dropped from FEATURES below.
# The ratio preserves their only unique signal: traffic vs industrial source fingerprint.

# ── Drop NaN rows introduced by lags / rolling ────────────────────────────────
fe.dropna(inplace=True)   # drops first ~12 rows (lag NaNs) + last row (no AQI_next)

new_cols = [c for c in fe.columns if c not in df.columns]
print(f'Shape after feature engineering : {fe.shape}')
print(f'NaN rows dropped                : {len(df) - len(fe)}')
print(f'New features added              : {new_cols}')
fe.head(3)

## 4. Feature Selection & Correlation Check

We expand the original feature set with the engineered features and verify correlation with AQI.

In [ ]:
# ── Lean 22-feature set (Tiers 1-3; Tier 4 dropped) ──────────────────────────
FEATURES = [
    # Tier 1 — AQI formula pollutants (direct causal drivers)
    'PM25', 'PM10', 'NO2', 'CO',

    # Tier 1 — Lag features (strongest autocorrelation)
    'AQI_lag1', 'AQI_lag2',
    'AQI_roll3_mean', 'AQI_roll6_mean',

    # Tier 2 — Remaining AQI formula pollutants
    'O3', 'SO2', 'NH3',

    # Tier 2 — Derived composites & cyclical time
    'NOx_total', 'hour_sin', 'hour_cos',

    # Tier 2 — Rolling trend + lag
    'AQI_roll12_mean', 'PM25_lag1',

    # Tier 2 — Humidity (directly inflates PM readings)
    'RH',

    # Tier 3 — Secondary meteorological & pollutants
    'NO', 'WS', 'AT', 'SR',

    # Tier 3 — VOC source fingerprint (replaces raw Benzene + Toluene)
    'Toluene_Benzene_ratio',
]

# ── Explicitly dropped (Tier 4) ───────────────────────────────────────────────
# Benzene    — raw; collinear with NOx, signal captured by Toluene_Benzene_ratio
# Toluene    — raw; same reason
# BP         — slow-changing; no signal within 48-h window
# AQI_lag3   — dominated by lag1 and rolling means
# PM10_lag1  — dominated by PM25_lag1 and rolling means
# month_sin/cos — seasonal signal weak vs diurnal at Anand Vihar bus terminal
# AQI_roll3_std — low marginal value over rolling means
# NOx (raw)  — replaced by NOx_total (engineered)

TARGET = 'AQI_next'   # next-hour AQI — what we actually want to forecast

missing = [f for f in FEATURES if f not in fe.columns]
if missing:
    print(f'WARNING — features not found and will be skipped: {missing}')

FEATURES = [f for f in FEATURES if f in fe.columns]
data = fe[FEATURES + [TARGET]].copy()

print(f'Features used  : {len(FEATURES)}')
print(f'Feature list   : {FEATURES}')
print(f'Target         : {TARGET}')
print(f'Data shape     : {data.shape}')

In [ ]:
# ── Correlation bar chart with AQI ───────────────────────────────────────────
corr = data.corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(14, 6), facecolor=DARK)
colors = [ACC1 if v >= 0 else ACC2 for v in corr.values]
ax.barh(corr.index[::-1], corr.values[::-1], color=colors[::-1], alpha=0.85)
ax.axvline(0, color=MUTED, linewidth=0.8, linestyle='--')
style_ax(ax, 'Feature Correlation with AQI', 'Pearson r', 'Feature')
plt.tight_layout()
plt.savefig('feature_correlation_anand_vihar.png', dpi=200, facecolor=DARK, bbox_inches='tight')
plt.show()
print('Top 10 features by |correlation|:')
print(corr.abs().sort_values(ascending=False).head(10).to_string())

## 5. Robust Scaling (StandardScaler)

Scale all features to zero mean and unit variance.  
A **separate scaler** is kept for the target (`AQI_next`) so predictions can be inverse-transformed back to real AQI values.  

> **Inference flow:** user supplies last 48 hours of pollutant + AQI readings → sequence fed to model → predicted AQI for the *next* hour is returned.

In [ ]:
feature_scaler = StandardScaler()
target_scaler  = StandardScaler()

scaled_features = feature_scaler.fit_transform(data[FEATURES])
scaled_target   = target_scaler.fit_transform(data[[TARGET]])   # TARGET = 'AQI_next'

scaled_data = np.hstack([scaled_features, scaled_target])

print(f'Scaled data shape : {scaled_data.shape}')
print(f'Feature range     : [{scaled_features.min():.4f}, {scaled_features.max():.4f}]')
print(f'Target range      : [{scaled_target.min():.4f}, {scaled_target.max():.4f}]')

## 6. Create Sliding Window Sequences

LSTM / GRU require 3-D input: **(samples, timesteps, features)**.  
A **48-hour** look-back is used (doubled from v1 based on EDA autocorrelation analysis showing significant correlation up to 48 lags).  

Each sequence covers hours `t-47 … t` (features) → predicts `AQI` at hour `t+1` (target).  
No future information leaks into the input: the model only sees what a user would already have.

In [ ]:
LOOK_BACK = 48   # EDA: AQI autocorrelation is significant up to ~48 h

def create_sequences(data, look_back, target_col_idx=-1):
    """
    data          : 2-D numpy array (timesteps × features+target)
    look_back     : number of past hours to use as input
    target_col_idx: column index of the target variable
    Returns X (3-D) and y (1-D)
    """
    X, y = [], []
    for i in range(look_back, len(data)):
        X.append(data[i - look_back:i, :target_col_idx])
        y.append(data[i, target_col_idx])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, LOOK_BACK)

print(f'X shape : {X.shape}  →  (samples, timesteps, features)')
print(f'y shape : {y.shape}  →  (samples,)')

## 7. Train / Validation / Test Split

**No shuffling** — time-series data must remain in chronological order.

| Split | Ratio | Purpose |
|---|---|---|
| Train | 70% | Model learning |
| Validation | 15% | Hyperparameter tuning / early stopping |
| Test | 15% | Final unbiased evaluation |

In [ ]:
n = len(X)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train, y_train = X[:train_end],        y[:train_end]
X_val,   y_val   = X[train_end:val_end], y[train_end:val_end]
X_test,  y_test  = X[val_end:],          y[val_end:]

print(f'Train      : {X_train.shape[0]:,} samples')
print(f'Validation : {X_val.shape[0]:,} samples')
print(f'Test       : {X_test.shape[0]:,} samples')
print(f'Input shape for models : {X_train.shape[1:]}')

## 8. Build LSTM Model

Architecture:  
`Input → BiLSTM(128) → Dropout(0.25) → LSTM(64) → BatchNorm → Dropout(0.2) → Dense(32, relu) → Dense(16, relu) → Dense(1)`

**Changes from v1:**
- Increased units (128/64) matching the expanded feature set
- Added `BatchNormalization` after second LSTM for training stability
- Slightly higher dropout (0.25) to combat overfitting on longer sequences

In [ ]:
def build_lstm(input_shape):
    model = Sequential([
        Bidirectional(LSTM(128, return_sequences=True), input_shape=input_shape),
        Dropout(0.25),

        LSTM(64, return_sequences=False),
        BatchNormalization(),
        Dropout(0.20),

        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1)
    ], name='BiLSTM_Model')

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    return model

lstm_model = build_lstm(input_shape=(X_train.shape[1], X_train.shape[2]))
lstm_model.summary()

## 9. Build GRU Model

Architecture:  
`Input → BiGRU(128) → Dropout(0.25) → GRU(64) → BatchNorm → Dropout(0.2) → Dense(32, relu) → Dense(16, relu) → Dense(1)`

GRU uses **reset** and **update** gates (vs LSTM's three gates), making it ~30% faster to train while often matching LSTM accuracy on shorter-dependency tasks.

In [ ]:
def build_gru(input_shape):
    model = Sequential([
        Bidirectional(GRU(128, return_sequences=True), input_shape=input_shape),
        Dropout(0.25),

        GRU(64, return_sequences=False),
        BatchNormalization(),
        Dropout(0.20),

        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1)
    ], name='BiGRU_Model')

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    return model

gru_model = build_gru(input_shape=(X_train.shape[1], X_train.shape[2]))
gru_model.summary()

## 10. Train Both Models

In [ ]:
EPOCHS     = 60
BATCH_SIZE = 64

def get_callbacks(model_name):
    return [
        EarlyStopping(monitor='val_loss', patience=8,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=4, min_lr=1e-6, verbose=1),
        ModelCheckpoint(f'best_{model_name}.keras', monitor='val_loss',
                        save_best_only=True, verbose=0)
    ]

print('═' * 50)
print('  Training LSTM …')
print('═' * 50)
lstm_history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=get_callbacks('lstm'),
    verbose=1
)

print('\n' + '═' * 50)
print('  Training GRU …')
print('═' * 50)
gru_history = gru_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=get_callbacks('gru'),
    verbose=1
)

print('\n✅ Both models trained')

In [ ]:
# ── Plot training curves for both models ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor=DARK)

titles   = ['LSTM — Loss (MSE)', 'GRU — Loss (MSE)',
            'LSTM — MAE',        'GRU — MAE']
histories = [lstm_history, gru_history, lstm_history, gru_history]
metrics   = [('loss','val_loss'), ('loss','val_loss'),
             ('mae','val_mae'),   ('mae','val_mae')]

for ax, hist, (tr_m, va_m), title in zip(axes.flat, histories, metrics, titles):
    ax.plot(hist.history[tr_m], color=ACC1, linewidth=2, label='Train')
    ax.plot(hist.history[va_m], color=ACC2, linewidth=2, label='Val')
    style_ax(ax, title, 'Epoch', tr_m.upper())
    ax.legend(facecolor=CARD, edgecolor='#2a2f4a', labelcolor=MUTED, fontsize=9)

plt.suptitle('Training History — LSTM vs GRU', color=WHITE, fontsize=13,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('training_history_anand_vihar.png', dpi=200, facecolor=DARK, bbox_inches='tight')
plt.show()

## 11. Evaluate & Compare on Test Set

In [ ]:
def evaluate_model(model, model_path, X_test, y_test, target_scaler, label):
    model.load_weights(model_path)
    y_pred_s = model.predict(X_test, verbose=0)
    y_pred = target_scaler.inverse_transform(y_pred_s).flatten()
    y_true = target_scaler.inverse_transform(y_test.reshape(-1,1)).flatten()

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100

    print(f'\n  {label}')
    print('  ' + '─' * 38)
    print(f'  MAE   : {mae:.4f}  AQI units')
    print(f'  RMSE  : {rmse:.4f}')
    print(f'  R²    : {r2:.4f}')
    print(f'  MAPE  : {mape:.2f}%')

    return y_true, y_pred, dict(label=label, MAE=mae, RMSE=rmse, R2=r2, MAPE=mape)

print('═' * 45)
print('  TEST SET EVALUATION')
print('═' * 45)

y_true_lstm, y_pred_lstm, metrics_lstm = evaluate_model(
    lstm_model, 'best_lstm.keras', X_test, y_test, target_scaler, 'BiLSTM')

y_true_gru, y_pred_gru, metrics_gru = evaluate_model(
    gru_model, 'best_gru.keras', X_test, y_test, target_scaler, 'BiGRU')

In [ ]:
# ── Metrics comparison table ──────────────────────────────────────────────────
comp_df = pd.DataFrame([metrics_lstm, metrics_gru]).set_index('label')
comp_df.index.name = 'Model'

print('\n  COMPARISON TABLE')
print('  ' + '─' * 38)
print(comp_df.round(4).to_string())

winner = comp_df['R2'].idxmax()
print(f'\n  🏆  Best R² → {winner}')

## 12. Visualise Predictions

In [ ]:
# ── 4-panel visualisation for each model ─────────────────────────────────────
def prediction_dashboard(y_true, y_pred, metrics, label, color, filename):
    fig, axes = plt.subplots(2, 2, figsize=(18, 10), facecolor=DARK)

    N = min(500, len(y_true))
    r2   = metrics['R2']
    mae  = metrics['MAE']
    rmse = metrics['RMSE']

    # 1. Time series
    ax = axes[0, 0]
    ax.plot(y_true[:N], color=ACC1,  linewidth=1.2, label='Actual',    alpha=0.85)
    ax.plot(y_pred[:N], color=color, linewidth=1.2, label='Predicted', alpha=0.85)
    style_ax(ax, f'{label} — Actual vs Predicted (first {N} hours)', 'Hours', 'AQI')
    ax.legend(facecolor=CARD, edgecolor='#2a2f4a', labelcolor=MUTED, fontsize=9)

    # 2. Scatter
    ax2 = axes[0, 1]
    ax2.scatter(y_true, y_pred, alpha=0.25, s=5, color=color, linewidths=0)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax2.plot(lims, lims, color=ACC2, linewidth=1.5, linestyle='--', label='Perfect fit')
    style_ax(ax2, f'Scatter  (R² = {r2:.4f})', 'Actual AQI', 'Predicted AQI')
    ax2.legend(facecolor=CARD, edgecolor='#2a2f4a', labelcolor=MUTED, fontsize=9)

    # 3. Residuals
    residuals = y_true - y_pred
    ax3 = axes[1, 0]
    ax3.plot(residuals[:N], color=ACC3, linewidth=0.8, alpha=0.7)
    ax3.axhline(0, color=ACC2, linewidth=1.2, linestyle='--')
    ax3.fill_between(range(N), residuals[:N], alpha=0.15, color=ACC3)
    style_ax(ax3, f'Residuals  (MAE={mae:.2f}, RMSE={rmse:.2f})', 'Hours', 'Error (AQI)')

    # 4. Residual distribution
    ax4 = axes[1, 1]
    ax4.hist(residuals, bins=60, color=color, alpha=0.55, edgecolor='none', density=True)
    xr = np.linspace(residuals.min(), residuals.max(), 200)
    ax4.plot(xr, sp_stats.norm.pdf(xr, residuals.mean(), residuals.std()),
             color=ACC2, linewidth=2, label='Normal curve')
    ax4.axvline(0, color=ACC3, linewidth=1.2, linestyle='--')
    style_ax(ax4, 'Residual Distribution', 'Error (AQI)', 'Density')
    ax4.legend(facecolor=CARD, edgecolor='#2a2f4a', labelcolor=MUTED, fontsize=9)

    plt.suptitle(f'{label} — Prediction Analysis', color=WHITE,
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(filename, dpi=200, facecolor=DARK, bbox_inches='tight')
    plt.show()

prediction_dashboard(y_true_lstm, y_pred_lstm, metrics_lstm,
                     'BiLSTM', ACC1, 'lstm_predictions_anand_vihar.png')

prediction_dashboard(y_true_gru,  y_pred_gru,  metrics_gru,
                     'BiGRU',  ACC4, 'gru_predictions_anand_vihar.png')

## 13. Classification Metrics (AQI Category)

CPCB AQI breakpoints:

| AQI Range | Category |
|-----------|----------|
| 0 – 50 | Good |
| 51 – 100 | Satisfactory |
| 101 – 200 | Moderate |
| 201 – 300 | Poor |
| 301 – 400 | Very Poor |
| 401+ | Severe |

In [ ]:
def aqi_to_category(aqi_values):
    cats = []
    for v in aqi_values:
        if   v <= 50:  cats.append('Good')
        elif v <= 100: cats.append('Satisfactory')
        elif v <= 200: cats.append('Moderate')
        elif v <= 300: cats.append('Poor')
        elif v <= 400: cats.append('Very Poor')
        else:          cats.append('Severe')
    return np.array(cats)

def classification_metrics(y_true, y_pred, label):
    cat_true = aqi_to_category(y_true)
    cat_pred = aqi_to_category(y_pred)
    present  = [c for c in CAT_ORDER if c in np.unique(cat_true)]

    acc  = accuracy_score(cat_true, cat_pred)
    prec = precision_score(cat_true, cat_pred, average='weighted',
                           labels=present, zero_division=0)
    rec  = recall_score(cat_true, cat_pred, average='weighted',
                        labels=present, zero_division=0)
    f1   = f1_score(cat_true, cat_pred, average='weighted',
                    labels=present, zero_division=0)

    print(f'\n  {label}')
    print('  ' + '─' * 40)
    print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1        : {f1:.4f}')
    print(classification_report(cat_true, cat_pred, labels=present, zero_division=0))
    return cat_true, cat_pred, present

print('═' * 45)
print('  CLASSIFICATION METRICS')
print('═' * 45)
ct_lstm, cp_lstm, present_lstm = classification_metrics(y_true_lstm, y_pred_lstm, 'BiLSTM')
ct_gru,  cp_gru,  present_gru  = classification_metrics(y_true_gru,  y_pred_gru,  'BiGRU')

In [ ]:
# ── Side-by-side confusion matrices ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6), facecolor=DARK)

for ax, (ct, cp, pres), title in zip(
    axes,
    [(ct_lstm, cp_lstm, present_lstm), (ct_gru, cp_gru, present_gru)],
    ['BiLSTM — Confusion Matrix (row-normalised)',
     'BiGRU  — Confusion Matrix (row-normalised)']
):
    cm = confusion_matrix(ct, cp, labels=pres)
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)

    im = ax.imshow(cm_norm, cmap='Blues', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(pres)))
    ax.set_yticks(range(len(pres)))
    ax.set_xticklabels(pres, rotation=30, ha='right', color=MUTED, fontsize=9)
    ax.set_yticklabels(pres, color=MUTED, fontsize=9)
    ax.set_xlabel('Predicted', color=MUTED, fontsize=9)
    ax.set_ylabel('Actual',    color=MUTED, fontsize=9)
    ax.set_title(title, color=WHITE, fontsize=11, fontweight='bold', pad=8)
    ax.set_facecolor(CARD)
    for spine in ax.spines.values(): spine.set_edgecolor('#2a2f4a')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(colors=MUTED)

    thresh = 0.5
    for i in range(len(pres)):
        for j in range(len(pres)):
            ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                    fontsize=8, color='black' if cm_norm[i,j] > thresh else WHITE)

plt.tight_layout()
plt.savefig('confusion_matrices_anand_vihar.png', dpi=200, facecolor=DARK, bbox_inches='tight')
plt.show()

## 14. Model Comparison Dashboard

In [ ]:
# ── Radar + bar comparison ───────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 8), facecolor=DARK)
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

# -- Bar chart: all metrics --------------------------------------------------
ax_bar = fig.add_subplot(gs[0])

metric_names = ['MAE', 'RMSE', 'R²', 'MAPE (%)']
lstm_vals = [metrics_lstm['MAE'], metrics_lstm['RMSE'],
             metrics_lstm['R2'],  metrics_lstm['MAPE']]
gru_vals  = [metrics_gru['MAE'],  metrics_gru['RMSE'],
             metrics_gru['R2'],   metrics_gru['MAPE']]

x = np.arange(len(metric_names))
w = 0.35
b1 = ax_bar.bar(x - w/2, lstm_vals, w, color=ACC1, alpha=0.8, label='BiLSTM')
b2 = ax_bar.bar(x + w/2, gru_vals,  w, color=ACC4, alpha=0.8, label='BiGRU')

for bar in list(b1) + list(b2):
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{bar.get_height():.3f}', ha='center', va='bottom',
                color=WHITE, fontsize=8)

ax_bar.set_xticks(x)
ax_bar.set_xticklabels(metric_names, color=MUTED)
style_ax(ax_bar, 'Metric Comparison — LSTM vs GRU', 'Metric', 'Value')
ax_bar.legend(facecolor=CARD, edgecolor='#2a2f4a', labelcolor=MUTED, fontsize=9)

# -- Overlay time-series (last 200 hours of test) ----------------------------
ax_ts = fig.add_subplot(gs[1])
N = 200
ax_ts.plot(y_true_lstm[-N:], color=WHITE,  linewidth=1.5, label='Actual',  alpha=0.9, zorder=3)
ax_ts.plot(y_pred_lstm[-N:], color=ACC1,   linewidth=1.2, label='LSTM',    alpha=0.8, zorder=2)
ax_ts.plot(y_pred_gru[-N:],  color=ACC4,   linewidth=1.2, label='GRU',     alpha=0.8, zorder=2,
           linestyle='--')
style_ax(ax_ts, f'Actual vs LSTM vs GRU (last {N} test hours)', 'Hours', 'AQI')
ax_ts.legend(facecolor=CARD, edgecolor='#2a2f4a', labelcolor=MUTED, fontsize=9)

plt.suptitle('Model Comparison Dashboard', color=WHITE, fontsize=14,
             fontweight='bold', y=1.02)
plt.savefig('model_comparison_anand_vihar.png', dpi=200, facecolor=DARK, bbox_inches='tight')
plt.show()

## 15. Save Models & Scalers

In [ ]:
import joblib

# Save both models
lstm_model.save('anandvihar_lstm_model.keras')
gru_model.save('anandvihar_gru_model.keras')

# Save scalers (required for inference)
joblib.dump(feature_scaler, 'anandvihar_feature_scaler.pkl')
joblib.dump(target_scaler,  'anandvihar_target_scaler.pkl')

print('✅ LSTM model  →  anandvihar_lstm_model.keras')
print('✅ GRU  model  →  anandvihar_gru_model.keras')
print('✅ Scalers     →  anandvihar_feature_scaler.pkl | anandvihar_target_scaler.pkl')

In [ ]:
# ── Inference helper ─────────────────────────────────────────────────────────
from tensorflow.keras.models import load_model

def predict_next_aqi(recent_df, model_path, feat_scaler, tgt_scaler,
                     features, look_back=48):
    """
    recent_df  : DataFrame with at least `look_back` rows and all feature columns.
    Returns    : predicted AQI (float) for the next hour.
    Input sequence must contain the FEATURES columns only (not AQI_next);
    the model predicts AQI_next from the last look_back rows.
    """
    model = load_model(model_path)
    scaled = feat_scaler.transform(recent_df[features].tail(look_back))
    x = scaled.reshape(1, look_back, len(features))
    pred_s = model.predict(x, verbose=0)
    return float(tgt_scaler.inverse_transform(pred_s)[0, 0])

# Quick sanity check using the last window from the test set
# Use last 48 rows of features; actual next-hour AQI is AQI_next of the last row
sample_df = data.iloc[-(LOOK_BACK + 1):-1]          # rows t-48 … t-1
actual_next = data['AQI_next'].iloc[-(LOOK_BACK + 1)]  # AQI at hour t (the true next hour)

for mpath, label in [('anandvihar_lstm_model.keras','LSTM'),
                     ('anandvihar_gru_model.keras', 'GRU')]:
    pred = predict_next_aqi(sample_df, mpath,
                            feature_scaler, target_scaler, FEATURES)
    print(f'{label:5s}  →  Predicted next-hour AQI: {pred:.2f}   Actual: {actual_next:.2f}')

---
## Summary

| Step | Detail |
|------|--------|
| Features | **22 features** (lean set — Tier 4 dropped after importance analysis) |
| Target | `AQI_next` = `AQI.shift(-1)` — true next-hour AQI, no leakage |
| Scaling | StandardScaler (mean=0, std=1) |
| Look-back window | **48 hours** (EDA-justified) |
| LSTM architecture | BiLSTM(128) -> LSTM(64) -> BN -> Dense(32->16->1) |
| GRU  architecture | BiGRU(128) -> GRU(64) -> BN -> Dense(32->16->1) |
| Optimizer | Adam (lr=0.001) |
| Loss | MSE |
| Train/Val/Test | 70% / 15% / 15% |
| Callbacks | EarlyStopping(8), ReduceLROnPlateau(4), ModelCheckpoint |

### Final Feature Set by Tier

| Tier | Features | Count |
|------|----------|-------|
| Tier 1 | PM25, PM10, NO2, CO, AQI_lag1, AQI_lag2, AQI_roll3_mean, AQI_roll6_mean | 8 |
| Tier 2 | O3, SO2, NH3, NOx_total, hour_sin, hour_cos, AQI_roll12_mean, PM25_lag1, RH | 9 |
| Tier 3 | NO, WS, AT, SR, Toluene_Benzene_ratio | 5 |
| Tier 4 (dropped) | Benzene, Toluene, BP, AQI_lag3, PM10_lag1, month_sin/cos, AQI_roll3_std | -- |

### Possible Further Improvements
- Attention mechanism over LSTM/GRU hidden states
- Transformer (Temporal Fusion Transformer) for multi-step forecasting
- Include WD (wind direction, sin/cos encoded) if available
- Multi-step ahead forecasting (2-h, 6-h, 12-h)
- Ensemble: average LSTM + GRU predictions
- Run permutation importance after training to validate tier assignments empirically